# Week 3 — Solutions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
plt.style.use("../../assets/mplstyle/course.mplstyle")


## Solution 1 — Two-model comparison

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "GBM": GradientBoostingClassifier(random_state=0),
}

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row, (name, m) in enumerate(models.items()):
    m.fit(Xtr, ytr)
    s = m.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(yte, s)
    axes[row, 0].plot(fpr, tpr, lw=2, label=f"AUC = {auc(fpr, tpr):.3f}")
    axes[row, 0].plot([0, 1], [0, 1], "k--", lw=0.7); axes[row, 0].legend()
    axes[row, 0].set(xlabel="FPR", ylabel="TPR", title=f"{name} — ROC")

    p, r, _ = precision_recall_curve(yte, s)
    axes[row, 1].plot(r, p, lw=2, label=f"AP = {average_precision_score(yte, s):.3f}")
    axes[row, 1].set(xlabel="recall", ylabel="precision", title=f"{name} — PR")
    axes[row, 1].legend()

    pp, pt = calibration_curve(yte, s, n_bins=10, strategy="quantile")
    axes[row, 2].plot(pp, pt, marker="o", lw=2)
    axes[row, 2].plot([0, 1], [0, 1], "k--", lw=0.7)
    axes[row, 2].set(xlabel="mean predicted", ylabel="observed",
                     title=f"{name} — Calibration")

plt.tight_layout(); plt.show()


**Reading.** GBM has slightly higher AUC/AP, but logistic regression is
**better calibrated** out of the box. For a clinical-style downstream task where the
probability is acted on, the calibrated logistic model is the safer ship; for a pure
ranking task, GBM wins.

## Solution 2 — Calibrate a random forest

In [ ]:
data = fetch_california_housing(as_frame=True)
X = data.data
y = (data.target > data.target.median()).astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

raw = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
cal = CalibratedClassifierCV(RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
                             method="isotonic", cv=5).fit(Xtr, ytr)

s_raw = raw.predict_proba(Xte)[:, 1]
s_cal = cal.predict_proba(Xte)[:, 1]

fig, ax = plt.subplots(figsize=(6, 5))
for name, s, c in [("raw RF", s_raw, "#D55E00"), ("isotonic RF", s_cal, "#0072B2")]:
    pp, pt = calibration_curve(yte, s, n_bins=10, strategy="quantile")
    ax.plot(pp, pt, marker="o", lw=2, label=name, color=c)
ax.plot([0, 1], [0, 1], "k--", lw=0.7)
ax.set(xlabel="mean predicted prob.", ylabel="observed positive fraction",
       title="Calibration: raw vs. isotonic random forest")
ax.legend(); plt.show()

from sklearn.metrics import roc_auc_score
print(f"AUC raw: {roc_auc_score(yte, s_raw):.4f}")
print(f"AUC cal: {roc_auc_score(yte, s_cal):.4f}")


**Reading.** The raw RF curve sags below the diagonal at high predicted
probabilities — classic random-forest under-confidence at the top end. Isotonic
calibration drags the curve onto the diagonal. The AUC changes only at the third
decimal — calibration **doesn't change discrimination**, it changes the
probabilities-as-probabilities. That is the whole point.
